In [1]:
#!pip install -U langchain
#!pip install -U langchain-core
#!pip install -U langchain-community
#!pip install -U langchain-google-genai


In [2]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import Tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_classic.memory import ConversationBufferMemory
#from langchain_community.memory import ConversationBufferMemory


# 🔐 Load API keys
load_dotenv(".env")
google_api_key=os.getenv("GOOGLE_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
)

# Simple QA tool (LLMChain)
qa_prompt = ChatPromptTemplate.from_template("Answer clearly: {question}")
qa_chain = qa_prompt | llm
qa_tool = Tool(
    name="Simple_QA",
    func=lambda question: qa_chain.invoke({"question": question}), 
    description="Answer the question without hallucination"
)

# Web search tool (Tavily)
tavily_search = TavilySearchResults(max_results=3)
search_tool = Tool(
    name="Web Search", 
    handle_parsing_errors=True,
    func=tavily_search.invoke,  # Fixed: changed .run to .invoke to stop tool crash
    description="Search the internet for current and live information"
)

# 🔧 Wrap all tools in an agent
tools = [qa_tool, search_tool]
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

C:\Users\user\AppData\Local\Temp\ipykernel_17452\3236287020.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import TavilySearchResults
C:\Users\user\AppData\Local\Temp\ipykernel_17452\3236287020.py:32: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(max_results=3)
C:\Users\user\AppData\Local\Temp\ipykernel_17452\3236287020.py:42: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be rem

In [3]:
#1) Zero-Shot ReAct (simple + reliable default)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True, 
    handle_parsing_errors=True
)
result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")

C:\Users\user\AppData\Local\Temp\ipykernel_17452\354406309.py:3: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(
C:\Users\user\AppData\Local\Temp\ipykernel_17452\354406309.py:10: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")
C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_conf



> Entering new AgentExecutor chain...
Question: Summarize LangChain in 2 lines, then tell me who created it.
Thought: Action: Web Search
Action Input: "LangChain" definition creatorThought: I need to summarize LangChain in 2 lines and state who created it.
Action: Web Search
Action Input: "LangChain" framework description summary
Observation: [{'title': 'What is the LangChain framework, and how can You use it to create your first application? | by Andrii Shchur | Data Science Collective | Medium', 'url': 'https://medium.com/data-science-collective/what-is-the-langchain-framework-and-how-can-you-use-it-to-create-your-first-application-eac11d3bc8b4', 'content': 'Firstly, let’s start with a brief description of LangChain. LangChain is a framework that facilitates various stages of LLM application development, from design to deployment, testing, and monitoring. LangChain provides convenient options for connecting LLMs, creating pipelines, monitoring, and checking the quality of applicati

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Final Answer: LangChain is an open-source framework designed to simplify the development of applications powered by large language models (LLMs). It provides modular tools and abstractions to easily connect LLMs with external data sources, prompts, and APIs. It was created by Harrison Chase.

> Finished chain.


In [4]:
#2) Conversational ReAct (with memory; non-chat model baseline)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")




> Entering new AgentExecutor chain...


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```
Thought: Do I need to use a tool? Yes
Action: Web Search
Action Input: who created LangChain and what is it
```
Observation: [{'title': 'LangChain - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/LangChain', 'content': '## History\n\n[edit]\n\nLangChain was launched in October 2022 as an open source project by Harrison Chase, while working at machine learning startup Robust Intelligence. In April 2023, LangChain had incorporated and the new startup raised over $20 million in funding at a valuation of at least $200 million from venture firm Sequoia Capital, a week after announcing a $10 million seed investment from Benchmark "Benchmark (venture capital firm)").\n\nIn the third quarter of 2023, the LangChain Expression Language (LCEL) was introduced, which provides a declarative way to define chains of actions.\n\nIn October 2023 LangChain introduced LangServe, a deployment tool to host LCEL code as a production-ready API. [...] | LangChain |\n\n| Developer | Harrison Chase |\n| R

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```
Thought: Do I need to use a tool? No
AI: LangChain is a software framework designed to facilitate the development of applications powered by large language models by providing standard interfaces for chains, agents, and prompts. It streamlines tasks like document analysis, chatbots, and retrieval-augmented generation.

LangChain was created by Harrison Chase in October 2022.
```

> Finished chain.


In [5]:
#3) Chat Conversational ReAct (recommended for GPT-3.5/4 chat)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    memory=memory,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True
)
result = agent.run("Plan a 3-step study path for LangChain.")
#result = agent.run("Summarize LangChain in 2 lines, then tell me who created it.")




> Entering new AgentExecutor chain...


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```json
{
    "action": "Final Answer",
    "action_input": "Here is a 3-step study path for learning LangChain:\n\n1. **Foundations & Core Concepts:** Start by understanding what Large Language Models (LLMs) are and the basic building blocks of LangChain. Learn about Prompts, Models (ChatModels and LLMs), and Output Parsers.\n2. **Chains & Retrieval-Augmented Generation (RAG):** Move on to connecting components together using Chains and LCEL (LangChain Expression Language). Focus heavily on RAG, learning how to load documents, split text, create embeddings, store them in vector databases, and retrieve context to answer questions.\n3. **Agents & Advanced Applications:** Learn how to build autonomous agents that use tools (like search engines or APIs) to complete complex tasks, and explore memory management and production deployment using LangSmith."
}
```

> Finished chain.


In [6]:
#from langchain.tools import StructuredTool
from langchain_classic.agents import AgentExecutor
#from langchain.agents import initialize_agent, AgentType    ---------->  initialize_agent is no more in Lang chain
from langchain_core.tools import StructuredTool
from pydantic import BaseModel
#from langchain import hub

# ✅ Define the input schema using Pydantic
class TitleInput(BaseModel):
    topic: str
    tone: str = "concise"

# ✅ Define the tool function
def title_tool_fn(topic: str, tone: str = "concise") -> str:
    return f"{tone.title()} Title: {topic} in Practice"

# ✅ Create the structured tool
title_tool = StructuredTool.from_function(
    name="TitleMaker",
    func=title_tool_fn,
    description="Generate a title given a topic and an optional tone.",
    args_schema=TitleInput
)


# ✅ Initialize the agent
agent = initialize_agent(
    tools=[title_tool],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# ✅ Run the agent
result = agent.run("Make a friendly title about LangGraph tutorials.")
print("✅ Result:", result)




> Entering new AgentExecutor chain...


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```
{
  "action": "TitleMaker",
  "action_input": {
    "topic": "LangGraph tutorials",
    "tone": "friendly"
  }
}
```
Observation: Friendly Title: LangGraph tutorials in Practice
Thought:

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```json
{
  "action": "Final Answer",
  "action_input": "Friendly Title: LangGraph tutorials in Practice"
}
```

> Finished chain.
✅ Result: Friendly Title: LangGraph tutorials in Practice


#  5 and 6 is for Open AI not Applicable for Gemini

In [8]:
#7) Self-Ask with Search (clarifies, then searches)

search_tool = Tool(
    name="Intermediate Answer",  # This name is REQUIRED
    func=TavilySearchResults(max_results=1).run,
    description="Use this tool to search for missing facts"
)

agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True
)
result = agent.run("When was the first LangChain release and who founded it?")




> Entering new AgentExecutor chain...


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Yes.
Follow up: When was the first LangChain release?
Intermediate answer: [{'title': 'LangChain - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/LangChain', 'content': '## History\n\n[edit]\n\nLangChain was launched in October 2022 as an open source project by Harrison Chase, while working at machine learning startup Robust Intelligence. In April 2023, LangChain had incorporated and the new startup raised over $20 million in funding at a valuation of at least $200 million from venture firm Sequoia Capital, a week after announcing a $10 million seed investment from Benchmark "Benchmark (venture capital firm)").\n\nIn the third quarter of 2023, the LangChain Expression Language (LCEL) was introduced, which provides a declarative way to define chains of actions.\n\nIn October 2023 LangChain introduced LangServe, a deployment tool to host LCEL code as a production-ready API. [...] | LangChain |\n\n| Developer | Harrison Chase |\n| Release | October 2022 |\n| Stable release | 0.1.16 / 1

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Are follow up questions needed here: Yes.
Follow up: Who founded LangChain?
Intermediate answer: [{'title': 'LangChain’s Harrison Chase on Building the Orchestration Layer for AI Agents | Sequoia Capital', 'url': 'https://sequoiacap.com/podcast/training-data-harrison-chase', 'content': 'Sonya Huang: Hi, and welcome to training data. We have with us today Harrison Chase, founder and CEO of LangChain. Harrison is a legend in the agent ecosystem, as the product visionary who first connected LLMs with tools and actions. And LangChain is the most popular agent building framework in the AI space. Today, we’re excited to ask Harrison about the current state of agents, the future potential and the path ahead. Harrison, thank you so much for joining us. And welcome to the show.\n\nHarrison Chase: Of course, thank you for having me.\n\n#### What are agents?', 'score': 0.82994765}]


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


So the final answer is: The first LangChain release was in October 2022, and it was founded by Harrison Chase.

> Finished chain.


![Model Diagram](agenttypes1.png)

![Model Diagram](agenttype2.png)

In [9]:
Deep Dive

SyntaxError: invalid syntax (774728019.py, line 1)

![Model Diagram](pro1.png)

![Model Diagram](pro2.png)

![Model Diagram](pro3.png)